# Responsible AI: Bias Analysis and Explainability

Two parts: a bias analysis of the imSitu dataset with a proposed individual fairness method, and Grad-CAM explanations for the waste classifier from `01_waste_classifier.ipynb`.

## 1. Bias in the imSitu dataset

#### The ImSitu dataset shows a gender bias in role representation, with "cooking" depicting 81 women and only 37 men, while "leading" features 52 men and just 20 women. This reinforces traditional stereotypes, women as caregivers and men as decision-makers. Such bias can influence AI models in hiring, content recommendations, and career guidance, leading to fewer leadership opportunities for women and discouraging men from domestic roles. Over time, this bias perpetuates unequal societal expectations, affecting both personal and professional growth.



## 2. Individual fairness method

### Proposed method: Fairness by Awareness

#### The "Fairness by Awareness" approach helps reduce gender bias in AI by making models aware of existing imbalances instead of ignoring them. In the ImSitu dataset, "cooking" is mostly shown with women (81 women, 37 men), while "leading" is mostly shown with men (52 men, 20 women). This reinforces stereotypes that affect hiring, career guidance, and recommendations. Rather than simply reweighting or adding data, this method trains models to recognize and correct bias using fairness-aware techniques. Post-processing adjustments ensure fair predictions, while continuous monitoring helps track and reduce bias over time. By acknowledging these disparities, this approach prevents AI from reinforcing stereotypes and ensures more balanced outcomes.

## 3. Grad-CAM on the waste classifier

I used Grad-CAM with xplique because it visually highlights the areas of the image that the model focuses on when classifying waste items. xplique is user-friendly and supports multiple explanation methods, making it an excellent choice for analyzing and understanding my model's decisions.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from xplique.attributions import GradCAM
import os

# Load your trained model
model = load_model("../models/model_it2.h5")  # saved by 01_waste_classifier.ipynb

# Class names in the model's output order. The LabelEncoder used in training
# sorts the classes alphabetically, so this list must follow the same order.
class_names = ['General Waste', 'Organic Waste', 'PMD', 'Paper & Cardboard']

# List of image paths you want to explain
image_paths = [
    "../data/dataset/General Waste/General Waste_100.png",
    "../data/dataset/PMD/PMD_70.png",
    "../data/dataset/Organic Waste/Organic Waste_99.png",
    "../data/dataset/Paper & Cardboard/Paper & Cardboard_42.png"
]

img_size = (128, 128)
gradcam = GradCAM(model)

# Loop through images
for img_path in image_paths:
    # Load and preprocess
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0

    # Predict class
    preds = model.predict(img_array)
    predicted_class = np.argmax(preds)
    confidence = preds[0][predicted_class]

    # Generate Grad-CAM
    explanations = gradcam.explain(img_array, targets=[predicted_class])
    heatmap = explanations[0].numpy()

    # Plot original image with heatmap
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    plt.axis('off')
    plt.title(f"{class_names[predicted_class]} ({confidence:.1%})")
    plt.show()


### I applied Grad-CAM to my image classifier model to understand how it makes decisions. The model was tested on 4 images of waste types. The heatmaps reveal which areas of the image the model considered most important.

> **Note:** the original run used a class list with PMD and Organic Waste swapped (fixed in the cell above), so two titles were wrong. In the model's real output order, image 2 (a PMD image) was predicted as PMD and image 3 (an organic waste image) as Organic Waste, so all four predictions were correct. The categories and the "mismatch" remarks in rows 2 and 3 below come from the swapped titles; re-run the cell above to regenerate the heatmaps with correct titles.

| Image Number | Waste Category      | Model Focus Area               | Explanation                                                                                         |
|--------------|---------------------|--------------------------------|-----------------------------------------------------------------------------------------------------|
| 1            | General Waste       | Center on phone and surroundings | The model focused on the central black object (a phone) and the general clutter around it, correctly associating it with non-recyclable general waste. |
| 2            | Organic Waste       | Juice carton and bottle top     | The model emphasized the juice carton’s cap and surrounding food remnants, though the classification may be slightly misleading given the visible plastic. |
| 3            | PMD                 | Food scraps and eggshells       | The heatmap covers the food waste prominently, suggesting a mismatch since PMD usually refers to plastics, metals, and drink cartons. |
| 4            | Paper & Cardboard   | Surface of paper textures       | Focus was on textured flat areas resembling cardboard, which aligns well with the waste classification. |

### Conclusion: The model shows promising attention patterns, though occasional misclassification (like organic waste on plastic bottles) indicates the need for improved training on overlapping waste types.



# Bias Analysis

### The model appears to focus on relevant features for each waste category. It consistently highlights key objects like packaging, food scraps, or paper textures, rather than being distracted by the background or lighting. There is no clear indication that the model favors one type of waste over others.

# Conclusion:

### The model performs well in identifying the important visual cues across different waste types. Its attention is directed toward the actual waste material instead of unrelated image elements, suggesting robustness to background and lighting variations. No significant bias was observed toward any particular category.